# Retail - Deploy Customer Churn Model to Hugging Face Hub

Pushes the retail customer churn model to the Hugging Face Hub as a
self-contained sklearn model repository with a model card, metadata,
and evaluation plots.

**Repository:** `ThabangTheActuaryCoder/retail-customer-churn-model`

**Prerequisites:**
- `huggingface-hub` installed on the cluster
- Databricks secret scope `huggingface` with key `token`, or `HF_TOKEN` env var
- Model artefacts present in `retail/artefacts/` (run `02_train_model` first)

**Steps:**
1. Authenticate with Hugging Face Hub
2. Create or retrieve the HF repository
3. Build a model card from the metadata JSON
4. Upload artefacts (model, metadata, plots, model card)
5. Print summary with link to the repository

In [ ]:
%run ../_setup

In [ ]:
import json
import os
import shutil
import tempfile
from pathlib import Path

import joblib
from huggingface_hub import HfApi, login

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
INDUSTRY = "retail"
MODEL_NAME = "customer_churn_model"
HF_REPO_ID = "ThabangTheActuaryCoder/retail-customer-churn-model"
ARTEFACTS_DIR = Path(_repo_root) / INDUSTRY / "artefacts"


def _get_hf_token():
    """Retrieve HF token from Databricks secrets or fall back to env var."""
    try:
        return dbutils.secrets.get(scope="huggingface", key="token")  # noqa: F821
    except Exception:
        return os.environ["HF_TOKEN"]


hf_token = _get_hf_token()
login(token=hf_token)

print(f"Industry       : {INDUSTRY}")
print(f"HF repo        : {HF_REPO_ID}")
print(f"Artefacts dir  : {ARTEFACTS_DIR}")

In [ ]:
# ---------------------------------------------------------------------------
# Create or retrieve the Hugging Face repository
# ---------------------------------------------------------------------------
api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, exist_ok=True)
print(f"Repository ready: https://huggingface.co/{HF_REPO_ID}")

In [ ]:
# ---------------------------------------------------------------------------
# Build model card (README.md) from metadata
# ---------------------------------------------------------------------------
metadata_path = ARTEFACTS_DIR / f"{MODEL_NAME}_metadata.json"
with open(metadata_path) as f:
    meta = json.load(f)

metrics = meta["metrics"]
features_all = meta["features_numeric"] + meta["features_categorical"]

model_card = f"""---
library_name: sklearn
tags:
- tabular-classification
- sklearn
- customer-churn
- retail
- south-africa
---

# Retail Customer Churn Model

A **{meta['classifier_type']}** pipeline for predicting customer churn,
trained on South African retail data.

## Intended Use

This model is intended for **educational and demonstration purposes** as part of
an end-to-end ML pipeline showcasing Databricks, MLflow, Azure ML, and
Hugging Face Hub integration.

## Model Details

| Property | Value |
|---|---|
| Classifier | `{meta['classifier_type']}` |
| Pipeline steps | {' -> '.join(meta['sklearn_pipeline_steps'])} |
| Training samples | {meta['train_samples']:,} |
| Test samples | {meta['test_samples']:,} |
| Target column | `{meta['target_column']}` |
| Created | {meta['created_at']} |

## Evaluation Metrics

| Metric | Score |
|---|---|
| Accuracy | {metrics['accuracy']:.4f} |
| Precision | {metrics['precision']:.4f} |
| Recall | {metrics['recall']:.4f} |
| F1 | {metrics['f1']:.4f} |
| ROC AUC | {metrics['roc_auc']:.4f} |

### Confusion Matrix

![Confusion Matrix](confusion_matrix.png)

### ROC Curve

![ROC Curve](roc_curve.png)

### Feature Importance

![Feature Importance](feature_importance.png)

## Features

**Numeric:** {', '.join(f'`{f}`' for f in meta['features_numeric'])}

**Categorical:** {', '.join(f'`{f}`' for f in meta['features_categorical'])}

## Sample Usage

```python
import joblib
from huggingface_hub import hf_hub_download
import pandas as pd

# Download and load the model
model_path = hf_hub_download(
    repo_id="{HF_REPO_ID}",
    filename="{MODEL_NAME}.joblib",
)
model = joblib.load(model_path)

# Create a sample input
sample = pd.DataFrame([{{{{
    {', '.join(f'"{{feat}}": 0' for feat in features_all)}
}}}}])

# Predict
prediction = model.predict(sample)
probabilities = model.predict_proba(sample)
print(f"Prediction: {{prediction}}, Probabilities: {{probabilities}}")
```
"""

print("Model card built successfully.")
print(f"  Classifier : {meta['classifier_type']}")
print(f"  Accuracy   : {metrics['accuracy']:.4f}")
print(f"  ROC AUC    : {metrics['roc_auc']:.4f}")

In [ ]:
# ---------------------------------------------------------------------------
# Upload artefacts to Hugging Face Hub
# ---------------------------------------------------------------------------
upload_dir = Path(tempfile.mkdtemp())

try:
    # Copy artefacts to the staging directory
    for fname in [
        f"{MODEL_NAME}.joblib",
        f"{MODEL_NAME}_metadata.json",
        "confusion_matrix.png",
        "roc_curve.png",
        "feature_importance.png",
    ]:
        src = ARTEFACTS_DIR / fname
        if src.exists():
            shutil.copy2(src, upload_dir / fname)
            print(f"  Staged: {fname}")
        else:
            print(f"  Skipped (not found): {fname}")

    # Write the model card
    (upload_dir / "README.md").write_text(model_card)
    print("  Staged: README.md")

    # Upload the folder
    api.upload_folder(
        folder_path=str(upload_dir),
        repo_id=HF_REPO_ID,
        commit_message=f"Upload {INDUSTRY} {MODEL_NAME} artefacts",
    )
    print(f"\nUpload complete: https://huggingface.co/{HF_REPO_ID}")
finally:
    shutil.rmtree(upload_dir, ignore_errors=True)

In [ ]:
# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
repo_url = f"https://huggingface.co/{HF_REPO_ID}"

print(f"\n{'='*60}")
print(f"  Retail Customer Churn - Hugging Face Deployment")
print(f"{'='*60}")
print(f"  Repository  : {HF_REPO_ID}")
print(f"  Classifier  : {meta['classifier_type']}")
print(f"  Accuracy    : {metrics['accuracy']:.4f}")
print(f"  ROC AUC     : {metrics['roc_auc']:.4f}")
print(f"{'='*60}")
print(f"\nView on Hugging Face:")
print(f"  {repo_url}")